### 00. IMPORT DE PACOTES E CONFIGURAÇÕES PADRÃO

In [13]:
import os
import requests
import logging
import zipfile
from datetime import datetime
from zoneinfo import ZoneInfo
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from time import perf_counter
from zoneinfo import ZoneInfo

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, regexp_replace, current_timestamp,
    from_utc_timestamp, to_date
)

from delta.tables import DeltaTable
from notebookutils import fs

# ---------------------------------------------------------------------
# Configuração de logging
# ---------------------------------------------------------------------
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

URL = "https://api-csvr.stg.cloud.cnj.jus.br/download_csv?tribunal=TJSP&indicador=&oj=&grau=&municipio=&ambiente=csv_p"

DATA_EXEC = datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y%m%d")

DESTINO_ZIP = f"/lakehouse/default/Files/DATAJUD_V2/ZIP/tjsp_{DATA_EXEC}.zip"
EXTRAIDO_DIR = f"/lakehouse/default/Files/DATAJUD_V2/EXTRAIDO/{DATA_EXEC}"
TABELA_PROCESSOS = "DOL_arqs_auxiliares.lista_processos_datajud"


StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 15, Finished, Available, Finished)

### 01. DOWNLOAD 

In [14]:
def download_zip(url, local_path, max_retries=5):
    logging.info("Etapa 1: Iniciando download do ZIP...")
    inicio = perf_counter()

    retry_strategy = Retry(
        total=max_retries,
        backoff_factor=5,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry_strategy))
    session.mount("http://", HTTPAdapter(max_retries=retry_strategy))

    headers = {"User-Agent": "Mozilla/5.0", "Referer": "https://justica-em-numeros.cnj.jus.br/"}
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    with session.get(url, headers=headers, stream=True, timeout=180) as response:
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))
        total_mb = total_size / 1024 / 1024
        logging.info(f"Tamanho total do arquivo: {total_mb:.2f} MB")

        downloaded = 0
        with open(local_path, "wb") as f:
            for i, chunk in enumerate(response.iter_content(chunk_size=1024 * 1024), 1):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    progresso = (downloaded / total_size) * 100 if total_size else 0
                    mb_baixado = downloaded / 1024 / 1024
                    logging.info(f"Chunk {i}: {progresso:.2f}% - {mb_baixado:.2f} MB baixados")

    logging.info(f"ZIP salvo em: {local_path}")
    logging.info(f"Tempo total download: {perf_counter() - inicio:.2f}s")


StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 16, Finished, Available, Finished)

### 02. EXTRAIR ZIP

In [15]:
def extrair_zip(zip_path, extract_dir):
    logging.info("Etapa 2: Extração do ZIP")
    inicio = perf_counter()

    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        arquivos = zip_ref.namelist()
        logging.info(f"Arquivos encontrados no ZIP: {len(arquivos)}")

        for i, arquivo in enumerate(arquivos, 1):
            zip_ref.extract(arquivo, extract_dir)
            caminho_extraido = os.path.join(extract_dir, arquivo)
            tamanho_mb = os.path.getsize(caminho_extraido) / 1024 / 1024
            progresso = (i / len(arquivos)) * 100
            logging.info(f"Extraído {i}/{len(arquivos)} ({progresso:.2f}%): {arquivo} ({tamanho_mb:.2f} MB)")

    logging.info(f"Extração concluída para: {extract_dir}")
    logging.info(f"Tempo extração: {perf_counter() - inicio:.2f}s")


StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 17, Finished, Available, Finished)

### 03. PROCESSAR E SALVAR TABELA NO DATALAKE

In [16]:
def processar_csvs_to_delta(input_dir, tabela):
    """
    Lê CSVs de um diretório no OneLake (EXTRAIDO_DIR),
    aplica limpeza/deduplicação e grava em uma tabela Delta com MERGE idempotente.
    Mantém apenas as colunas: Processo, data_download.
    """

    # Ajusta o caminho caso venha com /lakehouse/default/
    if input_dir.startswith("/lakehouse/default/"):
        input_dir = input_dir.replace("/lakehouse/default/", "")

    input_path = f"{input_dir}/*.csv"

    logging.info("Etapa 3: Leitura e processamento dos CSVs")
    logging.info(f"Lendo arquivos de: {input_path}")
    inicio = perf_counter()

    # -----------------------------------------------------------------
    # Leitura em Spark
    # -----------------------------------------------------------------
    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("delimiter", ";")
        .load(input_path)
    )

    if "Processo" not in df_raw.columns:
        logging.error("Coluna 'Processo' não encontrada.")
        return None

    # -----------------------------------------------------------------
    # Limpeza + deduplicação
    # -----------------------------------------------------------------
    df_proc = (
        df_raw
        .select(regexp_replace(col("Processo"), "[-.]", "").alias("Processo"))
        .filter(~col("Processo").rlike("(?i)sigiloso"))
        .dropDuplicates(["Processo"])
        .withColumn("data_ingestao", from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))
    )

    total_validos = df_proc.count()
    logging.info(f"Processos únicos lidos nesta execução: {total_validos:,}")

    # -----------------------------------------------------------------
    # Escrita idempotente via MERGE
    # -----------------------------------------------------------------
    if not spark._jsparkSession.catalog().tableExists(tabela):
        df_proc.write.format("delta").mode("overwrite").saveAsTable(tabela)
        logging.info(f"Tabela criada: {tabela}")
        novos = total_validos
    else:
        logging.info(f"Executando ingestão em {tabela}...")
        tgt = DeltaTable.forName(spark, tabela)

        count_before = tgt.toDF().count()

        (tgt.alias("t")
            .merge(df_proc.alias("s"), "t.Processo = s.Processo")
            .whenNotMatchedInsertAll()
            .execute())

        count_after = tgt.toDF().count()
        novos = count_after - count_before

        logging.info("MERGE finalizado.")

    logging.info(f"Processos acrescentados à tabela: {novos:,}")
    logging.info(f"Tempo total de processamento: {perf_counter() - inicio:.2f}s")

StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 18, Finished, Available, Finished)

### 04. EXECUÇÃO DAS FUNÇÕES

In [6]:
download_zip(URL, DESTINO_ZIP)

StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 8, Finished, Available, Finished)

2025-10-01 14:31:47,532 - INFO - Etapa 1: Iniciando download do ZIP...
2025-10-01 14:31:48,337 - INFO - Tamanho total do arquivo: 2807.12 MB
2025-10-01 14:31:49,277 - INFO - Chunk 1: 0.04% - 1.00 MB baixados
2025-10-01 14:31:49,382 - INFO - Chunk 2: 0.07% - 2.00 MB baixados
2025-10-01 14:31:49,813 - INFO - Chunk 3: 0.11% - 3.00 MB baixados
2025-10-01 14:31:49,910 - INFO - Chunk 4: 0.14% - 4.00 MB baixados
2025-10-01 14:31:49,980 - INFO - Chunk 5: 0.18% - 5.00 MB baixados
2025-10-01 14:31:50,046 - INFO - Chunk 6: 0.21% - 6.00 MB baixados
2025-10-01 14:31:50,123 - INFO - Chunk 7: 0.25% - 7.00 MB baixados
2025-10-01 14:31:50,198 - INFO - Chunk 8: 0.28% - 8.00 MB baixados
2025-10-01 14:31:50,273 - INFO - Chunk 9: 0.32% - 9.00 MB baixados
2025-10-01 14:31:50,345 - INFO - Chunk 10: 0.36% - 10.00 MB baixados
2025-10-01 14:31:50,426 - INFO - Chunk 11: 0.39% - 11.00 MB baixados
2025-10-01 14:31:50,505 - INFO - Chunk 12: 0.43% - 12.00 MB baixados
2025-10-01 14:31:50,581 - INFO - Chunk 13: 0.46% 

In [7]:
extrair_zip(DESTINO_ZIP, EXTRAIDO_DIR)

StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 9, Finished, Available, Finished)

2025-10-01 14:36:43,316 - INFO - Etapa 2: Extração do ZIP
2025-10-01 14:36:43,972 - INFO - Arquivos encontrados no ZIP: 21
2025-10-01 14:36:44,050 - INFO - Extraído 1/21 (4.76%): TJSP_tbl_correg.csv (0.45 MB)
2025-10-01 14:36:44,655 - INFO - Extraído 2/21 (9.52%): TJSP_CN_1-4.csv (36.36 MB)
2025-10-01 14:36:46,035 - INFO - Extraído 3/21 (14.29%): TJSP_CN_2-4.csv (168.72 MB)
2025-10-01 14:36:53,308 - INFO - Extraído 4/21 (19.05%): TJSP_CN_3-4.csv (1205.00 MB)
2025-10-01 14:37:05,566 - INFO - Extraído 5/21 (23.81%): TJSP_CN_4-4.csv (2019.14 MB)
2025-10-01 14:37:17,101 - INFO - Extraído 6/21 (28.57%): TJSP_CPL_1-4.csv (1939.41 MB)
2025-10-01 14:37:41,148 - INFO - Extraído 7/21 (33.33%): TJSP_CPL_2-4.csv (3749.25 MB)
2025-10-01 14:38:10,008 - INFO - Extraído 8/21 (38.10%): TJSP_CPL_3-4.csv (3033.05 MB)
2025-10-01 14:38:22,610 - INFO - Extraído 9/21 (42.86%): TJSP_CPL_4-4.csv (1853.18 MB)
2025-10-01 14:38:24,321 - INFO - Extraído 10/21 (47.62%): TJSP_CPL_15anos_1-4.csv (204.37 MB)
2025-10-0

In [17]:
processar_csvs_to_delta(EXTRAIDO_DIR, TABELA_PROCESSOS)

StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 19, Finished, Available, Finished)

2025-10-01 15:08:57,211 - INFO - Etapa 3: Leitura e processamento dos CSVs
2025-10-01 15:08:57,212 - INFO - Lendo arquivos de: Files/DATAJUD_V2/EXTRAIDO/20251001/*.csv
2025-10-01 15:10:17,744 - INFO - Processos únicos lidos nesta execução: 29,318,063
2025-10-01 15:10:54,886 - INFO - Tabela criada: DOL_arqs_auxiliares.lista_processos_datajud
2025-10-01 15:10:54,887 - INFO - Processos acrescentados à tabela: 29,318,063
2025-10-01 15:10:54,887 - INFO - Tempo total de processamento: 117.67s


In [19]:
df = spark.sql("SELECT * FROM DOL_arqs_auxiliares.lista_processos_datajud ORDER BY RAND() LIMIT 1000")
display(df)

StatementMeta(, 44fe3bb1-caa2-40d3-b7b1-41b69afec650, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1334e1b3-4f77-4505-abec-1dbe6a4f928d)